In [ ]:
!git clone https://github.com/alexafshar/aai501-grp1-project.git /content/aai501-grp1-project
%cd /content/aai501-grp1-project

Cloning into '/content/aai501-grp1-project'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 16 (delta 2), reused 16 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (16/16), 6.80 MiB | 8.67 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/aai501-grp1-project


In [ ]:
import pandas as pd

responses = pd.read_json("data/raw/response.jsonl", lines=True)
sources = pd.read_json("data/raw/source_info.jsonl", lines=True)

print("Chatbot answers:", len(responses))
print("Source records:", len(sources))

Chatbot answers: 17790
Source records: 2965


In [ ]:
answer = responses.iloc[0]
source_id = answer["source_id"]
matching_source = sources[sources["source_id"] == source_id].iloc[0]

print("Answer's source ID:", source_id)
print("Matching source's ID:", matching_source["source_id"])
print("\nChatbot answer:", answer["response"])
print("\nSource preview:", str(matching_source["source_info"])[:600])
print("\nReviewer markings:", answer["labels"])

Answer's source ID: 15592
Matching source's ID: 15592

Chatbot answer: The Anne Frank House has revealed that Anne Frank and her older sister, Margot, likely died at least a month earlier than previously believed. The sisters, who were imprisoned in Nazi concentration camps during the Holocaust, were thought to have died in March 1945, just two weeks before the Bergen-Belsen camp was liberated. However, new research examining archives from the Red Cross, the International Tracing Service, the Bergen-Belsen Memorial, and survivor testimonies suggests that the sisters did not survive until March. The exact dates of their deaths remain unclear, but it is thought that both had symptoms of typhus, the disease they succumbed to, before February 7, 1945.

Source preview: Seventy years ago, Anne Frank died of typhus in a Nazi concentration camp at the age of 15. Just two weeks after her supposed death on March 31, 1945, the Bergen-Belsen concentration camp where she had been imprisoned was lib

In [ ]:
merged = responses.merge(
    sources,
    on="source_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Rows after matching:", len(merged))
print("Answers without a source:", (merged["_merge"] == "left_only").sum())

Rows after matching: 17790
Answers without a source: 0


In [ ]:
marking_counts = merged["labels"].map(len)
merged["hallucinated"] = (marking_counts > 0).astype(int)

merged[["source_id", "labels", "hallucinated"]].head()

,source_id,labels,hallucinated
0,15592,[],0
1,15592,[],0
2,15592,"[{'start': 636, 'end': 653, 'text': 'February ...",1
3,15592,"[{'start': 693, 'end': 754, 'text': 'Anne's di...",1
4,15592,[],0


In [ ]:
excluded = merged["quality"].isin(["truncated", "incorrect_refusal"])
cleaned = merged.loc[~excluded].copy()

print("Removed:", excluded.sum())
print("Kept:", len(cleaned))

Removed: 173
Kept: 17617


In [ ]:
from pathlib import Path

output_path = Path("data/processed/responses_clean.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

cleaned.drop(columns="_merge").to_csv(output_path, index=False)

print("Rows saved:", len(cleaned))
print("Saved to:", output_path)

Rows saved: 17617
Saved to: data/processed/responses_clean.csv


In [ ]:
from google.colab import files

files.download(str(output_path))